In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test_sent_emo.csv
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/dev_sent_emo.csv
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/train_sent_emo.csv
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia123_utt11.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia1_utt6.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia126_utt1.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia97_utt7.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia184_utt2.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia182_utt7.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia232_utt5.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia17_utt24.wav
/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia96_utt5.wav
/kaggle/input/datasets/srushtivenkatesh/

In [2]:
!apt-get install -y ffmpeg
!pip install -q transformers torch accelerate jiwer librosa

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.0 MB/s eta 0:00:00:00:01


In [3]:
import os

# Your mounted dataset path on Kaggle
DATA_DIR = "/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data"

# Verify it's there
for split in ["train", "dev", "test"]:
    n = len([f for f in os.listdir(f"{DATA_DIR}/{split}") if f.endswith(".wav")])
    print(f"{split}: {n} wav files")

train: 9988 wav files
dev: 1112 wav files
test: 2747 wav files


In [4]:
import csv
import torch
import pandas as pd
import librosa
import IPython.display as ipd
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from jiwer import wer

OUTPUT_CSV = "/kaggle/working/asr_output.csv"   # Kaggle saves outputs here
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Device: {DEVICE}")

Device: cuda


In [5]:
train_df = pd.read_csv(f"{DATA_DIR}/train_sent_emo.csv")
dev_df   = pd.read_csv(f"{DATA_DIR}/dev_sent_emo.csv")
test_df  = pd.read_csv(f"{DATA_DIR}/test_sent_emo.csv")

print("Train shape:", train_df.shape)
print("Dev shape:  ", dev_df.shape)
print("Test shape: ", test_df.shape)
print("\nColumns:", train_df.columns.tolist())
print(train_df[["Dialogue_ID", "Utterance_ID", "Utterance", "Emotion"]].head(5))

Train shape: (9989, 11)
Dev shape:   (1109, 11)
Test shape:  (2610, 11)

Columns: ['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID', 'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime']
   Dialogue_ID  Utterance_ID  \
0            0             0   
1            0             1   
2            0             2   
3            0             3   
4            0             4   

                                           Utterance   Emotion  
0  also I was the point person on my companys tr...   neutral  
1                   You mustve had your hands full.   neutral  
2                            That I did. That I did.   neutral  
3      So lets talk a little bit about your duties.   neutral  
4                             My duties?  All right.  surprise  


In [6]:
def get_wav_path(split, dialogue_id, utterance_id):
    return os.path.join(DATA_DIR, split, f"dia{dialogue_id}_utt{utterance_id}.wav")

for split, dia, utt in [("train", 0, 0), ("train", 0, 1), ("test", 6, 1)]:
    path = get_wav_path(split, dia, utt)
    if os.path.exists(path):
        audio, sr = librosa.load(path, sr=16000)
        print(f"{path} | duration: {len(audio)/sr:.2f}s")
        display(ipd.Audio(path))
    else:
        print(f"Not found: {path}")

/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/train/dia0_utt0.wav | duration: 5.67s


/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/train/dia0_utt1.wav | duration: 1.47s


/kaggle/input/datasets/srushtivenkatesh/emotions2sdata/data/test/dia6_utt1.wav | duration: 2.60s


In [7]:
print("Loading Whisper large-v3...")
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True,
)
model.to(DEVICE)
processor = AutoProcessor.from_pretrained("openai/whisper-large-v3")

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=DTYPE,
    device=DEVICE,
    generate_kwargs={"language": "english", "task": "transcribe"},
)
print("Model loaded!")

Loading Whisper large-v3...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded!


In [9]:
!pip install soundfile

In [14]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────
!pip install -q openai-whisper jiwer soundfile tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 40.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:
# ── Cell 6: Load Whisper ──────────────────────────────────────────────────
import whisper
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

model = whisper.load_model("large-v3", device=DEVICE)
print("Model loaded!")

Device: cuda


100%|██████████████████████████████████████| 2.88G/2.88G [00:11<00:00, 266MiB/s]


Model loaded!


In [16]:
# ── Cell 7: Run ASR on all splits ─────────────────────────────────────────
import csv
import os
from tqdm.notebook import tqdm

def transcribe_split(split, df):
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=split):
        path = get_wav_path(split, row["Dialogue_ID"], row["Utterance_ID"])
        if not os.path.exists(path):
            continue

        try:
            result = model.transcribe(
                path,
                language="en",
                task="transcribe",
                fp16=torch.cuda.is_available(),  # fp16 only on GPU
            )
            text = result["text"].strip()
        except Exception as e:
            print(f"Error on {path}: {e}")
            text = ""

        results.append({
            "split":         split,
            "filename":      f"dia{row['Dialogue_ID']}_utt{row['Utterance_ID']}.wav",
            "dialogue_id":   row["Dialogue_ID"],
            "utterance_id":  row["Utterance_ID"],
            "emotion":       row["Emotion"],
            "reference":     row["Utterance"],
            "transcription": text,
        })

    print(f"[{split}] Done! {len(results)} files transcribed.")
    return results

all_results = []
all_results.extend(transcribe_split("train", train_df))
all_results.extend(transcribe_split("dev",   dev_df))
all_results.extend(transcribe_split("test",  test_df))

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)
print(f"Saved to {OUTPUT_CSV}")

train:   0%|          | 0/9989 [00:00<?, ?it/s]

[train] Done! 9988 files transcribed.


dev:   0%|          | 0/1109 [00:00<?, ?it/s]

[dev] Done! 1108 files transcribed.


test:   0%|          | 0/2610 [00:00<?, ?it/s]

[test] Done! 2610 files transcribed.
Saved to /kaggle/working/asr_output.csv


In [26]:
results_df = pd.DataFrame(all_results)

print("=" * 50)
print("WER RESULTS")
print("=" * 50)
for split in ["train", "dev", "test"]:
    subset   = results_df[results_df["split"] == split]
    split_wer = wer(subset["reference"].tolist(), subset["transcription"].tolist())
    print(f"{split:>6} WER: {split_wer:.4f} ({split_wer*100:.2f}%)")

print("\nWER by Emotion (test split):")
test_subset = results_df[results_df["split"] == "test"]
for emotion in sorted(test_subset["emotion"].unique()):
    emo_rows = test_subset[test_subset["emotion"] == emotion]
    emo_wer  = wer(emo_rows["reference"].tolist(), emo_rows["transcription"].tolist())
    print(f"  {emotion:<12} WER: {emo_wer:.4f} ({emo_wer*100:.2f}%)")

print("\nSample predictions (test split):")
print("-" * 50)
for _, row in results_df[results_df["split"] == "test"].head(5).iterrows():
    print(f"Emotion   : {row['emotion']}")
    print(f"Reference : {row['reference']}")
    print(f"Predicted : {row['transcription']}")
    print()

WER RESULTS
 train WER: 0.4558 (45.58%)
   dev WER: 0.4080 (40.80%)
  test WER: 0.5005 (50.05%)

WER by Emotion (test split):
  anger        WER: 0.4445 (44.45%)
  disgust      WER: 1.0834 (108.34%)
  fear         WER: 0.5534 (55.34%)
  joy          WER: 0.7254 (72.54%)
  neutral      WER: 0.4135 (41.35%)
  sadness      WER: 0.4418 (44.18%)
  surprise     WER: 0.5291 (52.91%)

Sample predictions (test split):
--------------------------------------------------
Emotion   : surprise
Reference : Why do all youre coffee mugs have numbers on the bottom?
Predicted : By the way, your coffee mugs have numbers on the bottom.

Emotion   : anger
Reference : Oh. Thats so Monica can keep track. That way if one on them is missing, she can be like, Wheres number 27?!
Predicted : Uh, that's so Monica can keep track. That way if one of them's missing, she can be like, where's number 27?

Emotion   : neutral
Reference : Y'know what?
Predicted : You know what?

Emotion   : neutral
Reference : Come on

In [30]:
import re
from jiwer import wer

def normalize(text):
    text = text.lower()
    
    # expand common contractions
    contractions = {
        "y'know": "you know",
        "gonna": "going to",
        "wanna": "want to",
        "gotta": "got to",
        "kinda": "kind of",
        "sorta": "sort of",
        "lotta": "lot of",
        "ain't": "is not",
        "won't": "will not",
        "can't": "cannot",
        "n't": " not",
        "'re": " are",
        "'ve": " have",
        "'ll": " will",
        "'m": " am",
        "'d": " would",
    }
    for k, v in contractions.items():
        text = text.replace(k, v)
    
    # ── NEW: remove filler words that Whisper often drops ──
    fillers = r"\b(uh|um|hmm|hm|mhm|aha|oh|ah|er|like|you know|i mean)\b"
    text = re.sub(fillers, "", text)

    # ── NEW: normalize numbers ──
    text = text.replace("1", "one").replace("2", "two").replace("3", "three")
    
    # remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

# then when computing WER:
references = [normalize(r) for r in subset["reference"].tolist()]
hypotheses = [normalize(h) for h in subset["transcription"].tolist()]
split_wer = wer(references, hypotheses)

In [31]:
results_df = pd.DataFrame(all_results)
print("=" * 50)
print("WER RESULTS")
print("=" * 50)

for split in ["train", "dev", "test"]:
    subset = results_df[results_df["split"] == split]
    refs = [normalize(r) for r in subset["reference"].tolist()]
    hyps = [normalize(h) for h in subset["transcription"].tolist()]
    split_wer = wer(refs, hyps)
    print(f"{split:>6} WER: {split_wer:.4f} ({split_wer*100:.2f}%)")

print("\nWER by Emotion (test split):")
test_subset = results_df[results_df["split"] == "test"]
for emotion in sorted(test_subset["emotion"].unique()):
    emo_rows = test_subset[test_subset["emotion"] == emotion]
    refs = [normalize(r) for r in emo_rows["reference"].tolist()]
    hyps = [normalize(h) for h in emo_rows["transcription"].tolist()]
    emo_wer = wer(refs, hyps)
    print(f"  {emotion:<12} WER: {emo_wer:.4f} ({emo_wer*100:.2f}%)")

WER RESULTS
 train WER: 0.3246 (32.46%)
   dev WER: 0.2708 (27.08%)
  test WER: 0.3742 (37.42%)

WER by Emotion (test split):
  anger        WER: 0.3148 (31.48%)
  disgust      WER: 0.9244 (92.44%)
  fear         WER: 0.4274 (42.74%)
  joy          WER: 0.5752 (57.52%)
  neutral      WER: 0.3031 (30.31%)
  sadness      WER: 0.3246 (32.46%)
  surprise     WER: 0.3645 (36.45%)


In [32]:
# inspect disgust samples
print("DISGUST SAMPLES:")
print("-" * 50)
disgust_rows = test_subset[test_subset["emotion"] == "disgust"]
for _, row in disgust_rows.head(10).iterrows():
    print(f"Reference : {row['reference']}")
    print(f"Predicted : {row['transcription']}")
    print()

DISGUST SAMPLES:
--------------------------------------------------
Reference : Ugh, can you believe that guy!
Predicted : Can you believe that guy?

Reference : Oh wait, Joey, you cant go like that! You stink!
Predicted : Oh, what? Joey, you can't go like that. You stink.

Reference : Ewww! Oh! Its the Mattress King!
Predicted : Ew, oh, it's the Mattress King!

Reference : Dont look honey. Change the channel! Change the channel!
Predicted : Boo! Don't look, honey. Change the channel.

Reference : Four percent. Okay. I tip more than that when theres a bug in my food.
Predicted : I tip more than that when there's a bug in my food.

Reference : Yeah, Im gonna go to a doctor who went to school in a mini-mall.
Predicted : Yeah, I'm gonna go to a doctor who went to medical school at a mini mall.

Reference : You know what, where he hugs you and kinda rolls you away and... Oh... my....God.
Predicted : You know, like where he hugs you and then he kind of rolls you away and... Oh. My. God

In [29]:
print(test_subset["emotion"].value_counts())

emotion
neutral     1256
joy          402
anger        345
surprise     281
sadness      208
disgust       68
fear          50
Name: count, dtype: int64
